In [1]:
%pip install osmnx geopandas pandas
%pip install psycopg2-binary
%pip install sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings("ignore")

# Conection
host = 'localhost'
port = '5433'
database = 'layereddb'
schema='berlin_source_data'
user_name='peter_finger'
password='tY3uG1hM8vD6fP5sL'

#connection to db after you opened tunnel
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

In [3]:
# Conection
host = 'localhost'
port = '5433'
database = 'layereddb'
schema='berlin_source_data'
user_name='peter_finger'
password='tY3uG1hM8vD6fP5sL'

#connection to db after you opened tunnel
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

In [4]:
# import libraries
import osmnx as ox
import geopandas as gpd
import pandas as pd

In [5]:
# ✅ Enables caching: Speeds up repeated queries
# 🖥 Logs details to the console (helpful for debugging)
ox.settings.use_cache = True
ox.settings.log_console = True

In [6]:
create_table_sql = '''
CREATE TABLE IF NOT EXISTS atms (
    id VARCHAR(20) PRIMARY KEY,
    district_id VARCHAR(20) NOT NULL,
    name VARCHAR(200) NOT NULL DEFAULT 'Unknown',
    address VARCHAR(200),
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    geometry VARCHAR,
    neighborhood VARCHAR(100),
    district VARCHAR(100),
    atm_type VARCHAR(100),
    neighborhood_id VARCHAR(20),
    accessibility VARCHAR(100),
    fee VARCHAR(100),
    opening_hours VARCHAR(100),
    CONSTRAINT district_id_fk FOREIGN KEY (district_id)
        REFERENCES districts(district_id)
        ON DELETE RESTRICT
        ON UPDATE CASCADE
);
'''



In [7]:
# Load the CSV file into a Pandas DataFrame
from sqlalchemy import text

# Read CSV
df_atms = pd.read_csv('atm_cleaned.csv',dtype={'neighborhood_id':str,'district_id':str,'id':str})

# Display the first 5 rows of the DataFrame
print("First 5 rows of the df_atms DataFrame:")
print(df_atms.head())

# Display the column names and their data types
print("\nColumn information for df_atms DataFrame:")
df_atms.info()

# Query Postgres information_schema for the 'atms' table column metadata
query = """
SELECT table_schema, column_name, data_type, is_nullable, column_default,
       character_maximum_length, numeric_precision, numeric_scale, ordinal_position
FROM information_schema.columns
WHERE table_name = 'atms'
ORDER BY ordinal_position;
"""

try:
    with engine.connect() as conn:
        df_cols = pd.read_sql(text(query), conn)
    print("\nPostgres information_schema columns for 'atms':")
    if df_cols.empty:
        print("No 'atms' table found in the connected database.")
    else:
        print(df_cols.to_string(index=False))
except Exception as e:
    print(f"Error querying database for 'atms' metadata: {e}")

# Summary statistics and null counts
import numpy as np

print("\nSummary statistics (numeric):")
try:
    print(df_atms.describe(include=[np.number]).T)
except Exception as e:
    print(f"Error computing numeric summary: {e}")

print("\nSummary statistics (all):")
try:
    print(df_atms.describe(include='all').T)
except Exception as e:
    print(f"Error computing full summary: {e}")

print("\nNull counts per column:")
print(df_atms.isna().sum().to_string())

print("\nNull percentages per column:")
print((df_atms.isna().mean()*100).round(2).to_string())



First 5 rows of the df_atms DataFrame:
          id opening_hours  fee                       geometry   latitude  \
0   60852951          24/7  NaN  POINT (13.4656184 52.5246695)  52.524670   
1   78252154           NaN  NaN  POINT (13.3986266 52.5237445)  52.523744   
2   87036263   06:00-23:00  NaN  POINT (13.3842822 52.5329853)  52.532985   
3  213106623           NaN  NaN  POINT (13.4411367 52.5421697)  52.542170   
4  239694634           NaN  NaN  POINT (13.5216425 52.4975011)  52.497501   

   longitude    atm_type accessibility  \
0  13.465618  standalone    accessible   
1  13.398627  standalone    accessible   
2  13.384282      indoor    accessible   
3  13.441137      indoor    accessible   
4  13.521643  standalone    accessible   

                                     address                       name  \
0             Storkower Bogen, 10369, Berlin         Berliner Volksbank   
1    Oranienburger Straße, 13, 10178, Berlin  Bank Für Sozialwirtschaft   
2   Caroline-Michael

In [9]:
df_atms.to_sql('atms', schema='berlin_source_data', con=engine, if_exists='append', index=False)

62

In [10]:
print("Properties of the 'atms' table:")


qry = text("""
        SELECT *
        FROM berlin_source_data.atms
        """)
with engine.connect() as conn:
            props = pd.read_sql(qry, conn)


Properties of the 'atms' table:


In [11]:
props.head()

,id,opening_hours,fee,geometry,latitude,longitude,atm_type,accessibility,address,name,district,neighborhood_id,neighborhood,district_id
0,60852951,24/7,None,POINT (13.4656184 52.5246695),52.524670,13.465618,standalone,accessible,"Storkower Bogen, 10369, Berlin",Berliner Volksbank,Lichtenberg,1111,Fennpfuhl,11011011
1,78252154,None,None,POINT (13.3986266 52.5237445),52.523744,13.398627,standalone,accessible,"Oranienburger Straße, 13, 10178, Berlin",Bank Für Sozialwirtschaft,Mitte,101,Mitte,11001001
2,87036263,06:00-23:00,None,POINT (13.3842822 52.5329853),52.532985,13.384282,indoor,accessible,"Caroline-Michaelis-Straße, 10115, Berlin",Sparda-Bank Berlin,Mitte,101,Mitte,11001001
3,213106623,None,None,POINT (13.4411367 52.5421697),52.542170,13.441137,indoor,accessible,"Greifswalder Straße, 87-88, 10409, Berlin",Sparkasse,Pankow,301,Prenzlauer Berg,11003003
4,239694634,None,None,POINT (13.5216425 52.4975011),52.497501,13.521643,standalone,accessible,"Otto-Schmirgal-Straße, 3, 10319, Berlin",Berliner Volksbank,Lichtenberg,1101,Friedrichsfelde,11011011


In [12]:
print("Properties of the 'atms' table:")

if 'df_cols' in globals() and isinstance(df_cols, (pd.DataFrame,)):
    print(df_cols.to_string(index=False))
else:
    try:
        qry = text("""
        SELECT column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_name = 'atms'
        ORDER BY ordinal_position;
        """)
        with engine.connect() as conn:
            props = pd.read_sql(qry, conn)
        if props.empty:
            print("No 'atms' table found in the database.")
        else:
            print(props.to_string(index=False))
    except Exception as e:
        print(f"Could not query information_schema: {e}")
        print("\nFallback: df_atms.info() (if loaded in notebook)")
        try:
            df_atms.info()
        except Exception as e2:
            print(f"df_atms not available: {e2}")

Properties of the 'atms' table:
      table_schema     column_name        data_type is_nullable column_default character_maximum_length  numeric_precision  numeric_scale  ordinal_position
berlin_source_data              id           bigint         YES           None                     None               64.0            0.0                 1
berlin_source_data   opening_hours             text         YES           None                     None                NaN            NaN                 2
berlin_source_data             fee             text         YES           None                     None                NaN            NaN                 3
berlin_source_data        geometry             text         YES           None                     None                NaN            NaN                 4
berlin_source_data        latitude double precision         YES           None                     None               53.0            NaN                 5
berlin_source_data       longitu